In [1]:
import requests
import json
import io
import boto3
import boto3.s3.transfer
import os 

webproxy = os.environ.get("WEBPROXY", "")
bucket = os.environ["TNGRI_S3_BUCKET_NAME"]

params = {
    'y1': 1992, 'y2': 2025, 'publicationId': 5, 'datasetId': 8, 'measureId': -1
}
resp = requests.get(webproxy + 'http://www.cbr.ru/dataservice/data', params=params)
resp_json = resp.json()


# write to buffer
file = io.BytesIO()
for row in resp_json['RawData']:
    row_json = json.dumps(row)
    file.write(row_json.encode('utf-8'))
    file.write(b'\n')
file.seek(0)


# write to s3
config = boto3.s3.transfer.TransferConfig(use_threads=False)
s3 = boto3.client('s3')
s3.upload_fileobj(file, bucket, 'Stage/test.json', Config=config)

res = s3.list_objects(Bucket=bucket, Prefix='Stage/test.json')
res['Contents']

In [1]:
SELECT *
FROM read_json('test.json') 
